In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_PATH = Path("data/genomics")

In [ ]:
meta = pd.read_csv(PROJECT_PATH / "reference/metadata_complete.csv")
meta['population'] = meta['Strain'] + '_' + meta['Culture'].astype(str).str.zfill(2)
meta

In [ ]:
dfs = []

for i, row in meta.iterrows():
    df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
    df["Strain"] = row['Strain']
    df["Culture"] = row['Culture']
    df["Day"] = int(row['Day'])
    dfs.append(df)

df = pd.concat(dfs)
df


In [ ]:
select_lineage = "PCr"
pops = meta.query(f'Strain=="{select_lineage}"')['population'].unique()
timepoints = sorted(meta.query(f'Strain=="{select_lineage}"')['Day'].unique())
print(pops)
print(timepoints)

In [ ]:
def traceAlleleFreq(pop, min_freq=0.33):

    D = []
    sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')
    for i, row in sorted_meta.iterrows():
        df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        D.append(df)

    # Remove mutations detected in the wild-type background
    # 1. Select background mutations with strong signal
    wt_background=D[0].loc[D[0].frequency>0.01, 'position']
    #print(f"WT background mutations: {wt_background.values}")
    D_=[]

    #print(f"# of mutations\t# of (freq>{min_freq}):")
    for i in range(len(D)):
        # 2. Filter out mutations by position on the chromosome
        df=D[i][~D[i]['position'].isin(wt_background)]
        D_.append(df)
        #print(f"{df.shape[0]}\t{sum(df['frequency']>min_freq)}")
    
    tracked_positions=[]
    for i in range(len(D_)):
        d=D_[i]
        tracked_positions.extend(list(d.loc[ (d['frequency']>min_freq), 'position' ])) #& ~d['mutation_category'].isin(['mobile_element_insertion','large_deletion']), 'position' ] ))
    tracked_positions=set(tracked_positions)
    print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")

    ## 3. Trace the frequency of those mutations
    T=[]
    for pos in tracked_positions:

        freq=[]
        for i in range(len(D_)):
            d = D_[i]
            ind = d['position']==pos
            
            if sum(ind)<1:
                freq.append(0)
            else:
                freq.append( (d.loc[ind , 'frequency'].values)[0] )
                gene_name = d.loc[ind, 'gene_name'].values[0]
                gene_product = d.loc[ind, 'gene_product'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                new_seq = d.loc[ind, 'new_seq'].values[0]
                codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0]
                aa_pos = d.loc[ind, 'aa_position'].values[0]
                gene_pos = d.loc[ind, 'gene_position'].values[0]
                mut_cat = d.loc[ind, 'mutation_category'].values[0]

        T.append({'position': pos, 'freq': np.round(freq,2), 
                'gene_name':gene_name, 'gene_product':gene_product,
                'aa_ref_seq':aa_ref_seq, 'aa_new_seq':aa_new_seq,
                'aa_pos': aa_pos, 'gene_pos':gene_pos, 'mut_cat': mut_cat,
                'new_seq': new_seq, 'codon_ref_seq': codon_ref_seq})

    T=pd.DataFrame(T)
    T.fillna({'aa_ref_seq': '',
              'aa_new_seq': '',
              'aa_pos': ''},inplace=True)
    
    nsi = T['aa_pos']==''
    T.loc[nsi,'label'] = T.loc[nsi,'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
    T.loc[~nsi, 'label'] = T.loc[~nsi,'gene_name'] + ' ' + T.loc[~nsi,'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0','',regex=True) + T.loc[~nsi,'aa_new_seq']
        # T.loc[nsi,'gene_pos'] + ' '
    T.sort_values(by=['mut_cat','gene_name'],ascending=False,inplace=True)
    T.reset_index(inplace=True,drop=True)
    
    return T

In [ ]:
af=[]
for pop in pops:
    # print(traceAlleleFreq(pop, min_freq=0.2).shape)
    af.append(traceAlleleFreq(pop, min_freq=0.1))
    af[-1].to_csv(f"data/genomics/data/processed/traced_alleles/{select_lineage}/{pop}.csv", index=False)

In [ ]:
culture = 1
cult = 4

af[culture-1].loc[
    (af[culture-1]['gene_name'] == 'acrR') & (af[culture-1]['gene_pos'] == 'coding (237-244/648 nt)'),
    'label'
] = 'acrR IS1 insertion'
#

af[cult-1].loc[
    (af[cult-1]['gene_name'] == 'acrR') & (af[cult-1]['gene_pos'] == 'coding (404-411/648 nt)'),
    'label'
] = 'acrR IS1 insertion'


In [ ]:
c = 2
cu = 3
clt = 5
cltr = 6

af[c-1].loc[
    (af[c-1]['gene_name'] == 'acrR') & (af[c-1]['gene_pos'] == 'coding (265-268/648 nt)'),
    'label'
] = 'acrR IS5 insertion'

af[cu-1].loc[
    (af[cu-1]['gene_name'] == 'acrR') & (af[cu-1]['gene_pos'] == 'coding (265-268/648 nt)'),
    'label'
] = 'acrR IS5 insertion'

af[clt-1].loc[
    (af[clt-1]['gene_name'] == 'acrR') & (af[clt-1]['gene_pos'] == 'coding (265-268/648 nt)'),
    'label'
] = 'acrR IS5 insertion'

af[cltr-1].loc[
    (af[cltr-1]['gene_name'] == 'acrR') & (af[cltr-1]['gene_pos'] == 'coding (265-268/648 nt)'),
    'label'
] = 'acrR IS5 insertion'




In [ ]:
all = []
for ix, df in enumerate(af):
    dff = df.copy()
    dff['Pop'] = ix+1 
    all.append(dff)
combined_df=pd.concat(all)


combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])
combined_df


In [ ]:
common = np.concatenate([df.label.unique() for df in af])
common, counts = np.unique(common, return_counts=True)
common = pd.DataFrame(dict(zip(common, counts)), index=[0]).melt().sort_values(by='value', ascending=False)
common = common.query('value>0')
print(common['variable'].tolist())
f, ax = plt.subplots(1, 1, figsize=(24, 4))
sns.barplot(x=common['variable'], y=common['value'], hue=common['variable'], palette="deep", ax=ax)
# rotate x axis labels 90 degrees
plt.xticks(rotation=90);
plt.xticks(fontsize=20);
plt.yticks(fontsize=20);
#ax3.set_ylabel("Qualitative")
# common
#f.savefig(PROJECT_PATH / "figures/mutation_counts/PCr_mutations.png", dpi=300, bbox_inches='tight')

In [ ]:
# List of labels to filter by
labels_of_interest = [
    'acrR mobile_element_insertion', 'ftsI Q536L', 'ompC mobile_element_insertion',
    'acrB Q569L', 'cpxA T252P', 'marR R94H', 'ftsI|ftsO L|NA376|NAS|NA',
    'lysO/aqpZ snp_intergenic', 'envZ N278Y', 'cpxA G96S', 'aroK small_indel',
    'acrB I626F', 'acrB N68D', 'acrB A916G', 'ftsI A317V', 'ftsH W106R',
    'fliI E290D', 'envZ V241G', 'envZ P148S', 'insG/nanX snp_intergenic',
    'ftsI|ftsO D|NA409|NAG|NA', 'ftsI I336S', 'marR R27P', 'ldtA R133H',
    'marR small_indel', 'ompC small_indel', 'rpoB G536V', 'rpoC T1310A',
    'ygfB P184Q', 'ynfE T387A'
]

# Filter the DataFrame by the specified labels
filtered_df = combined_df[combined_df['label'].isin(labels_of_interest)]

# Get unique gene_name values
unique_gene_names = filtered_df['gene_name'].unique()

# Print the unique gene names
print(unique_gene_names)



#

['ftsI|ftsO' # yes
 'marR' # yes
 'ftsI' # yes
 'fliI' 
 'envZ' # yes
 'acrB'# yes
   'insG/nanX' 
   'ompC' 
   'acrR'# yes
 'cpxA' 
 'lysO/aqpZ' #
 'ygfB' #yes
 'rpoB' # yes
   'ldtA' #
   'ynfE'#
     'rpoC' # yes
     'ftsH' # yes
     'aroK'] # yes


In [ ]:
import matplotlib.colors as mcolors
plt.rcParams["font.family"] = "Nimbus Roman"


# Assuming `af` is a list of DataFrames with the following structure
# Example structure for af: [{'label': '...', 'gene_name': '...', 'freq': [..]}]
# af = [...]
standard_map = plt.cm.get_cmap('Purples')

# Create a new colormap that interpolates between white and the 'Greens' colormap
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # Replace the first color (for 0 values) with white
custom_map = mcolors.ListedColormap(new_colors)

# Define your labels of interest
labels_of_interest = [
    'acrR IS5 insertion', 
    'acrR IS1 insertion',
    'ftsI Q536L',
    'ompC mobile_element_insertion',
    'acrB Q569L',
    'cpxA T252P',
    'marR R94H',
    'ftsI|ftsO L|NA376|NAS|NA',
    'envZ N278Y',
    'cpxA G96S', 
    'aroK small_indel',
    'acrB I626F',
    'acrB N68D', 
    'acrB A916G', 
    'ftsI A317V', 
    'ftsH W106R',
    'fliI E290D',
    'envZ V241G',
    'envZ P148S', 
    'ftsI|ftsO D|NA409|NAG|NA',
    'ftsI I336S', 
    'marR R27P', 
    'ldtA R133H',
    'marR small_indel', 
    'ompC small_indel', 
    'rpoB G536V', 
    'rpoC T1310A',
    'ygfB P184Q', 
    'lysO/aqpZ snp_intergenic',
    'ygfB P184Q',
    'acrAB N68D',
    'insG/nanX snp_intergenic',
    'ynfE T387A'
]

manual_label_order = [
    # acrR / acrB / acrAB
    'acrR IS5 insertion',
    'acrR IS1 insertion',
    'acrB Q569L',
    'acrB I626F',
    'acrB N68D',
    'acrB A916G',
    'acrAB N68D',

    # ftsI / ftsO
    'ftsI Q536L',
    'ftsI A317V',
    'ftsI I336S',
    'ftsI|ftsO L|NA376|NAS|NA',
    'ftsI|ftsO D|NA409|NAG|NA',

    # ompC
    'ompC mobile_element_insertion',
    'ompC small_indel',

    # cpxA / envZ / marR
    'cpxA T252P',
    'cpxA G96S',
    'envZ N278Y',
    'envZ V241G',
    'envZ P148S',
    'marR R94H',
    'marR R27P',
    'marR small_indel',

   
    
    'ynfE T387A',
    

    # other
    
    'fliI E290D',
    'ldtA R133H',
    'rpoB G536V',
    'rpoC T1310A',
    'aroK small_indel',
    'ftsH W106R',
    'ygfB P184Q',
    'lysO/aqpZ snp_intergenic',
    'insG/nanX snp_intergenic',
]



# Define alternate labels for the selected mutations
alternate_labels = {
    'acrR IS5 insertion': r'$\it{acrR}$ IS5 insertion',
    'acrR IS1 insertion': r'$\it{acrR}$ IS1 insertion',
    'ftsI Q536L': 'FtsI Q536L',
    'ompC mobile_element_insertion': r'$\it{ompC}$ IS1 insertion',
    'acrB Q569L': 'AcrB Q569L',
    'cpxA T252P': 'CpxA T252P',
    'marR R94H': 'MarR R94H',
    'ftsI|ftsO L|NA376|NAS|NA': 'FtsI L376X',
    'envZ N278Y': 'EnvZ N278Y',
    'cpxA G96S': 'CpxA G96S',
    'aroK small_indel': r'$\it{aroK}$ indel',
    'acrB I626F': 'AcrB I626F',
    'acrB N68D': 'AcrB N68D',
    'acrB A916G': 'AcrB A916G',
    'ftsI A317V': 'FtsI A317V',
    'ftsH W106R': 'FtsH W106R',
    'fliI E290D': 'FliI E290D',
    'envZ V241G': 'EnvZ V241G',
    'envZ P148S': 'EnvZ P148S',
    'ftsI|ftsO D|NA409|NAG|NA': 'FtsI D409X',
    'ftsI I336S': 'FtsI I336S',
    'marR R27P': 'MarR R27P',
    'ldtA R133H': 'LdtA R133H',
    'marR small_indel': r'$\it{marR}$ indel',
    'ompC small_indel': r'$\it{ompC}$ indel',
    'rpoB G536V': 'RpoB G536V',
    'rpoC T1310A': 'RpoC T1310A',
    'ygfB P184Q': 'YgfB P184Q',
    'lysO/aqpZ snp_intergenic': r'$\it{lysO/aqpZ}$ snp intergenic',
    'acrAB N68D': 'AcrAB N68D',
    'insG/nanX snp_intergenic': r'$\it{insG/nanX}$ snp intergenic',
    'ynfE T387A': 'YnfE T387A'
}


# Filter the DataFrame for these labels
#filtered_df = combined_df[combined_df['label'].isin(labels_of_interest)]
filtered_df = combined_df  # No filtering

# Map alternate labels for the selected mutations
filtered_df['label'] = filtered_df['label'].replace(alternate_labels)

# Pivot the DataFrame to have 'label' as rows, 'Pop' as columns, and 'last_freq' as values
heatmap_data = filtered_df.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# Apply manual label ordering based on mapped labels
ordered_labels = [alternate_labels.get(lbl, lbl) for lbl in manual_label_order if alternate_labels.get(lbl, lbl) in heatmap_data.index]

# Reorder heatmap data according to manual order
heatmap_data = heatmap_data.loc[ordered_labels]
freq_mat = heatmap_data.values
sorted_labels = heatmap_data.index

# Timepoints (Pop values) for y-axis
timepoints = heatmap_data.columns

# Create figure and axes
fig, ax = plt.subplots(figsize=(2 + freq_mat.shape[0] // 2, 6))
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# Create the heatmap
sns.heatmap(
    freq_mat.T,
    yticklabels=timepoints,
    xticklabels=sorted_labels,
    cmap=custom_map,
    vmin=0,
    vmax=1,
    ax=ax,
    annot=annot_matrix,
    fmt="",
    annot_kws={'size': 20, 'ha': 'center', 'va': 'center'},
    cbar=False,
    linewidths=.5,
    linecolor='lightgrey'
)

# Style adjustments
ax.set_ylabel('Culture Number', fontsize=25)
ax.set_xlabel('', fontsize=25)
# ax.set_title('High Frequency Mutations of MG$^{{\\mathrm{{CEF-R}}}}$', fontsize=25)

ax.tick_params(axis='both', which='major', labelsize=15, length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color("black")

plt.tight_layout()
plt.show()

fig.savefig(PROJECT_PATH / "figures/final/PCr_full.png", dpi=300, bbox_inches='tight')


In [ ]:


## FULL
filtered_df = combined_df
# Pivot the DataFrame to have 'label' as rows, 'Pop' as columns, and 'freq' as values
heatmap_data = filtered_df.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# Convert the DataFrame to a numpy array for sorting
freq_mat = heatmap_data.values

# Sort the rows by the sum of frequencies descending
sort_index = np.argsort(freq_mat.sum(axis=1))[::-1]  # Sort in descending order
freq_mat = freq_mat[sort_index]  # Reorder frequency matrix based on sorted index
sorted_labels = heatmap_data.index[sort_index]  # Get the sorted labels

# Timepoints (populations) for y-axis
timepoints = heatmap_data.columns

# Create a figure and axes with a size based on the number of rows/labels
fig, ax = plt.subplots(figsize=(2 + freq_mat.shape[0] // 2, 5))

# Create the heatmap with custom settings, including larger font sizes for ticks and annotations
sns.heatmap(freq_mat.T, yticklabels=timepoints, xticklabels=sorted_labels, cmap='rocket_r', vmin=0, vmax=1, ax=ax,
            annot=True, fmt=".2f", annot_kws={'size': 12, 'ha': 'center', 'va': 'center'}, cbar=False)

# Set labels and title with larger font sizes
ax.set_ylabel('Culture Number', fontsize=14)  # Set y-axis label with larger font
ax.set_xlabel('Mutation', fontsize=14)  # Set x-axis label with larger font
ax.set_title(' Mutation Frequencies Across Populations', fontsize=16)  # Set title with larger font

# Adjust tick label font sizes
ax.tick_params(axis='both', which='major', labelsize=12)

# Rotate x-axis labels for readability with larger font sizes
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)

# Display the plot
plt.tight_layout()
plt.show()

fig.savefig(PROJECT_PATH / "figures/mutation_heatmaps/PA_full.png", dpi=300, bbox_inches='tight')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming `af` is a list of DataFrames with the following structure
# Example structure for af: [{'label': '...', 'gene_name': '...', 'freq': [..]}]
# af = [...]

# Define the gene names to include (set to None or empty list for no filtering)
designated_gene_names = ['fusA', 'trkH']  # Example gene names to include; set to [] for no filtering

# Extract unique labels and their counts from the data
all_labels = np.concatenate([df['label'].unique() for df in af])
unique_labels, counts = np.unique(all_labels, return_counts=True)

# Create a DataFrame with labels and counts
label_counts = pd.DataFrame({'label': unique_labels, 'count': counts})

# Map gene_name and the last freq value to labels
gene_name_map = {row['label']: row['gene_name'] for df in af for _, row in df.iterrows()}
freq_map = {row['label']: row['freq'][-1] for df in af for _, row in df.iterrows()}  # Assuming 'freq' is a list or array
label_counts['gene_name'] = label_counts['label'].map(gene_name_map)
label_counts['freq'] = label_counts['label'].map(freq_map)

# Define a mapping of old labels to new alternate labels
label_mapping = {
    'mrcB|mrcB T|T702|657S|S': 'mrcB T702S',
    'mgtL/mgtA small_indel': 'mgtL/mgtA indel',
    'trkH small_indel': 'trkH indel',
    'yicC mobile_element_insertion': 'yicC IS2 insertion'
}

# Apply the label mapping to the label_counts DataFrame
label_counts['label'] = label_counts['label'].replace(label_mapping)

# Apply the label mapping to combined_df to match the updated labels
combined_df['label'] = combined_df['label'].replace(label_mapping)

# Optional: Filter `label_counts` and `combined_df` if designated_gene_names is not empty
if designated_gene_names:
    label_counts = label_counts[label_counts['gene_name'].isin(designated_gene_names)]
    combined_df = combined_df[combined_df['gene_name'].isin(designated_gene_names)]

# Separate labels by gene_name
fusA_labels = label_counts[label_counts['gene_name'] == 'fusA']
trkH_labels = label_counts[label_counts['gene_name'] == 'trkH']
other_labels = label_counts[~label_counts['gene_name'].isin(['fusA', 'trkH'])]

# Sort `fusA` and `trkH` groups by frequency
fusA_labels = fusA_labels.sort_values(by='freq', ascending=True)
trkH_labels = trkH_labels.sort_values(by='freq', ascending=False)

# Sort other labels by last freq value (descending)
other_labels = other_labels.sort_values(by='freq', ascending=False)

# Combine the groups: fusA first, trkH second, others last
label_counts_sorted = pd.concat([fusA_labels, trkH_labels, other_labels], ignore_index=True)

# Create a categorical order for labels
label_order = label_counts_sorted['label'].tolist()

# Filter the main DataFrame to include only the sorted labels
filtered_df = combined_df[combined_df['label'].isin(label_order)]

# Pivot the DataFrame to have 'label' as rows, 'Pop' as columns, and 'last_freq' as values
heatmap_data = filtered_df.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# Reindex the heatmap data to match the label order
heatmap_data = heatmap_data.reindex(label_order)

# Convert the DataFrame to a numpy array for sorting
freq_mat = heatmap_data.values
timepoints = heatmap_data.columns

# Create a figure and axes with a size based on the number of rows/labels
fig, ax = plt.subplots(figsize=(6, 8))
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# Create the heatmap with custom settings, including inverted colors
sns.heatmap(freq_mat.T, yticklabels=timepoints, xticklabels=label_order, cmap='Blues', vmin=0, vmax=1, ax=ax,
            annot=annot_matrix, fmt="", annot_kws={'size': 20, 'ha': 'center', 'va': 'center'}, cbar=False,
            linewidths=.5,  # Thickness of the grid lines
            linecolor='lightgrey'  # Color of the grid lines
)

# Set labels and title with larger font sizes
ax.set_ylabel('Culture Number', fontsize=25)  # Set y-axis label with larger font
ax.set_xlabel('Mutation', fontsize=25)  # Set x-axis label with larger font
ax.set_title('High Frequency Mutations of MG$^{{\\mathrm{{AMI}}}}$', fontsize=25)  # Set title with larger font

# Adjust tick label font sizes
ax.tick_params(axis='both', which='major', labelsize=15)

# Rotate x-axis labels for readability with larger font sizes
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

for spine in ax.spines.values():
    spine.set_visible(True)  # Make spines visible
    spine.set_linewidth(2)   # Set the border width
    spine.set_color("black") # Set the border color

# Display the plot
plt.tight_layout()
plt.show()

# Save the figure
fig.savefig(PROJECT_PATH / "figures/mutation_heatmaps/1205-PC_fusA+trkH_sorted.png", dpi=300, bbox_inches='tight')
